# inspect_soft_prompt_surface
Establishes, from source, what OmniGen2 ships for soft prompting (nothing), where learnable tokens could be injected, and what the token budget allows. Companion to `inspect_official_training`, which does the same for the LoRA path.

Two candidate injection points, following the advisor's (i)/(ii) split:
(i) learnable embeddings at the MLLM input — classic prompt tuning, the setting Petrov analyses;
(ii) learnable tokens appended to the conditioning stream that reaches the diffusion transformer — a post-MLLM diffusion-conditioning soft prompt conceptually related to Textual Inversion (Gal et al. 2022), generalised from a concept to a domain rather than sharing its mechanism, which learns a token embedding inside the text-encoder path.

Sections 5-7 print the exact lines each variant would have to modify, and section 8 computes the parameter budget against LoRA rank 4. Read-only: clones the repo, reads two config files from the Hub, writes nothing.

## 1. Clone (skip if present)

In [ ]:
import os, subprocess, sys, re, textwrap
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","--depth","1","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
print(subprocess.run(["git","-C",REPO,"rev-parse","--short","HEAD"], capture_output=True, text=True).stdout.strip(), "= repo commit")

18e6f9d = repo commit


## 2. Print helpers

In [ ]:
def head(t): print("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)

def show(relpath, first=None, last=None, grep=None, ctx=2):
    p = os.path.join(REPO, relpath)
    if not os.path.exists(p):
        print(f"  [absent] {relpath}"); return []
    lines = open(p, encoding="utf-8").read().split("\n")
    if grep:
        hits = [i for i, l in enumerate(lines) if re.search(grep, l)]
        for i in hits:
            for j in range(max(0, i - ctx), min(len(lines), i + ctx + 1)):
                print(f"  {relpath}:{j+1:>4}  {lines[j]}")
            print("  " + "-" * 60)
        return hits
    sel = lines[(first or 1) - 1:(last or len(lines))]
    for k, l in enumerate(sel, start=(first or 1)):
        print(f"  {relpath}:{k:>4}  {l}")
    return list(range(len(sel)))

## 3. The complete adaptation surface that ships
Everything OmniGen2 provides for parameter-efficient adaptation.

In [ ]:
head("options/ — every training config in the repo")
for f in sorted(os.listdir(os.path.join(REPO, "options"))):
    print(f"  options/{f}")
head("train.py — every branch that freezes or adds parameters")
show("train.py", grep=r"lora_ft|add_adapter|requires_grad_|LoraConfig", ctx=1)
head("scripts/train/ — launch scripts")
for f in sorted(os.listdir(os.path.join(REPO, "scripts", "train"))):
    print(f"  scripts/train/{f}")


options/ — every training config in the repo
  options/ft.yml
  options/ft_lora.yml

train.py — every branch that freezes or adds parameters
  train.py:  44  
  train.py:  45  from peft import LoraConfig
  train.py:  46  
  ------------------------------------------------------------
  train.py: 257      
  train.py: 258      args.train.lora_ft = args.train.get('lora_ft', False)
  train.py: 259      if args.train.lora_ft:
  ------------------------------------------------------------
  train.py: 258      args.train.lora_ft = args.train.get('lora_ft', False)
  train.py: 259      if args.train.lora_ft:
  train.py: 260          model.requires_grad_(False)
  ------------------------------------------------------------
  train.py: 259      if args.train.lora_ft:
  train.py: 260          model.requires_grad_(False)
  train.py: 261  
  ------------------------------------------------------------
  train.py: 264          # now we will add new LoRA weights the transformer layers
  train.py: 26

## 4. Absence of a soft-prompt path
Searches the whole repo for any prompt-tuning, prefix-tuning or textual-inversion implementation. The assertion is the finding: the arm has no official recipe and must be built.

In [ ]:
PATTERNS = ["soft.?prompt", "prompt.?tuning", "prefix.?tuning", "textual.?inversion",
            "virtual.?token", "learnable.?token", "p.?tuning", "inputs_embeds"]
hits = {}
for root, dirs, files in os.walk(REPO):
    dirs[:] = [d for d in dirs if d not in {".git", "__pycache__", "assets", "example_images"}]
    for fn in files:
        if not fn.endswith((".py", ".yml", ".yaml", ".sh", ".md")): continue
        p = os.path.join(root, fn)
        try: txt = open(p, encoding="utf-8", errors="ignore").read()
        except Exception: continue
        for pat in PATTERNS:
            for m in re.finditer(pat, txt, re.I):
                hits.setdefault(pat, []).append(os.path.relpath(p, REPO))

head("soft-prompt / prefix-tuning / textual-inversion search")
for pat in PATTERNS:
    where = sorted(set(hits.get(pat, [])))
    print(f"  {pat:22s} {len(where)} file(s)  {where[:4] if where else ''}")
implemented = {f for pat in PATTERNS[:-1] for f in hits.get(pat, [])}
assert not implemented, f"expected no shipped soft-prompt implementation, found {implemented}"
print("\n  CONFIRMED: no prompt-tuning, prefix-tuning or textual-inversion implementation ships with OmniGen2.")
print("  The only parameter-efficient path is LoRA (train.py lora_ft + options/ft_lora.yml).")
print("  `inputs_embeds` hits, if any, are in vendored library code, not in an OmniGen2 training path.")


soft-prompt / prefix-tuning / textual-inversion search
  soft.?prompt           0 file(s)  
  prompt.?tuning         0 file(s)  
  prefix.?tuning         0 file(s)  
  textual.?inversion     0 file(s)  
  virtual.?token         0 file(s)  
  learnable.?token       0 file(s)  
  p.?tuning              0 file(s)  
  inputs_embeds          0 file(s)  

  CONFIRMED: no prompt-tuning, prefix-tuning or textual-inversion implementation ships with OmniGen2.
  The only parameter-efficient path is LoRA (train.py lora_ft + options/ft_lora.yml).
  `inputs_embeds` hits, if any, are in vendored library code, not in an OmniGen2 training path.


## 5. Injection point (i) — MLLM input
The pipeline tokenises the prompt and calls the MLLM on token **ids**. Prompt tuning replaces those ids with embeddings, so this call is the line that has to change.

In [ ]:
head("pipeline_omnigen2.py :: _get_qwen2_prompt_embeds")
show("omnigen2/pipelines/omnigen2/pipeline_omnigen2.py", grep=r"def _get_qwen2_prompt_embeds", ctx=0)
show("omnigen2/pipelines/omnigen2/pipeline_omnigen2.py", grep=r"self\.mllm\(", ctx=6)
print(textwrap.dedent("""
  What variant (i) requires:
    the call passes text_input_ids, so learnable vectors cannot be supplied as-is.
    Qwen2 accepts inputs_embeds; the change is to embed the ids with the MLLM's
    input embedding table, concatenate p learnable rows, and call with inputs_embeds
    plus a matching attention mask. The MLLM stays frozen; only the p rows train.
    The learnable rows pass through all MLLM layers, so they can change how the
    instruction itself is interpreted. This is the setting Petrov analyses.
"""))


pipeline_omnigen2.py :: _get_qwen2_prompt_embeds
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 274      def _get_qwen2_prompt_embeds(
  ------------------------------------------------------------
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 318              logger.warning(
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 319                  "The following part of your input was truncated because Gemma can only handle sequences up to"
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 320                  f" {max_sequence_length} tokens: {removed_text}"
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 321              )
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 322  
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 323          prompt_attention_mask = text_inputs.attention_mask.to(device)
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 324          prompt_embeds = self.mllm(
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 325              text

## 6. Injection point (ii) — conditioning stream to the diffusion transformer
The MLLM's last hidden states are what the diffusion decoder consumes. Appending learnable rows to that tensor bypasses the MLLM entirely.

In [ ]:
head("pipeline_omnigen2.py :: what _get_qwen2_prompt_embeds returns, and where it goes")
show("omnigen2/pipelines/omnigen2/pipeline_omnigen2.py", grep=r"return prompt_embeds, prompt_attention_mask", ctx=3)
show("omnigen2/pipelines/omnigen2/pipeline_omnigen2.py", grep=r"prompt_embeds, prompt_attention_mask = self\._get_qwen2_prompt_embeds", ctx=3)
head("transformer_omnigen2.py :: the text features the decoder receives")
show("omnigen2/models/transformers/transformer_omnigen2.py", grep=r"text_feat_dim", ctx=1)
print(textwrap.dedent("""
  What variant (ii) requires:
    a concatenation onto prompt_embeds (and its attention mask) after the MLLM call.
    The learnable rows never pass through the MLLM, so they cannot reinterpret the
    instruction; they add conditioning the decoder attends to alongside the text.
    This is a post-MLLM diffusion-conditioning soft prompt, conceptually related to
    Textual Inversion but not its mechanism: Textual Inversion learns a token embedding
    inside the text-encoder path. Generalised here from one concept to a domain. Petrov's analysis does not cover it: his result is about tokens
    prepended to the transformer whose attention is being studied.
"""))


pipeline_omnigen2.py :: what _get_qwen2_prompt_embeds returns, and where it goes
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 336  
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 337          prompt_embeds = prompt_embeds.to(dtype=dtype, device=device)
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 338  
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 339          return prompt_embeds, prompt_attention_mask
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 340      
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 341      def _apply_chat_template(self, prompt: str):
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 342          prompt = [
  ------------------------------------------------------------
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 445                  batch_size * num_images_per_prompt, -1
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 446              )
  omnigen2/pipelines/omnigen2/pipeline_omnigen2.py: 447  
  omnigen2/pipeline

## 7. Token budget
Training truncates text at `maximum_text_tokens`; inference truncates at `max_sequence_length`. Both bound how many learnable tokens variant (i) can prepend, and — if the decoder is to stay within the sequence lengths it was trained on — variant (ii) as well.

In [ ]:
head("training cap")
show("options/ft_lora.yml", grep=r"maximum_text_tokens", ctx=1)
show("train.py", grep=r"OmniGen2Collator\(", ctx=1)
show("omnigen2/dataset/omnigen2_train_dataset.py", grep=r"max_length=self\.max_token_len", ctx=4)
head("inference cap")
show("omnigen2/pipelines/omnigen2/pipeline_omnigen2.py", grep=r"max_sequence_length: int = ", ctx=0)
show("omnigen2/pipelines/omnigen2/pipeline_omnigen2.py", grep=r"truncation=True", ctx=3)
print(textwrap.dedent("""
  Training caps text at 888 tokens with truncation=True; the shipped inference default
  is 1024. The released model therefore operates 15% above its training cap already.
  888 is the largest length the decoder has actually seen; 1024 is the operating cap
  this thesis uses everywhere (asserted in the few-shot arm). Neither is a hard limit
  in code -- both are arguments -- but exceeding 1024 puts text conditioning outside
  anything the model was trained or released to handle.
"""))


training cap
  options/ft_lora.yml:   9    use_chat_template: true
  options/ft_lora.yml:  10    maximum_text_tokens: 888
  options/ft_lora.yml:  11    prompt_dropout_prob: !!float 0.0001
  ------------------------------------------------------------
  train.py: 363          drop_last=True,
  train.py: 364          collate_fn=OmniGen2Collator(tokenizer=text_tokenizer, max_token_len=args.data.maximum_text_tokens)
  train.py: 365      )
  ------------------------------------------------------------
  omnigen2/dataset/omnigen2_train_dataset.py: 185  
  omnigen2/dataset/omnigen2_train_dataset.py: 186          text_inputs = self.tokenizer(
  omnigen2/dataset/omnigen2_train_dataset.py: 187              instruction,
  omnigen2/dataset/omnigen2_train_dataset.py: 188              padding="longest",
  omnigen2/dataset/omnigen2_train_dataset.py: 189              max_length=self.max_token_len,
  omnigen2/dataset/omnigen2_train_dataset.py: 190              truncation=True,
  omnigen2/dataset/omnig

## 8. Parameter budget against LoRA rank 4
Reads the MLLM hidden size, layer count and grouped-query K/V widths from the released config rather than assuming them, then measures the instruction lengths that compete for the same budget. The deep variant's per-token cost is `2 x num_key_value_heads x head_dim x layers`, not `2 x hidden_size x layers`: on this model K and V are 256 wide against a 2048 hidden state, so the naive figure is 8x too large. `soft_prompt_iii_probe` derives the same number and is the reference.

In [ ]:
from google.colab import drive
from transformers import AutoConfig, AutoTokenizer
import json as _json, math
drive.mount("/content/drive")

MODEL = "OmniGen2/OmniGen2"
cfg = AutoConfig.from_pretrained(MODEL, subfolder="mllm", trust_remote_code=True)
H = getattr(cfg, "hidden_size", None) or cfg.text_config.hidden_size
L = getattr(cfg, "num_hidden_layers", None) or cfg.text_config.num_hidden_layers
_tc = getattr(cfg, "text_config", cfg)
NH  = getattr(_tc, "num_attention_heads", None) or cfg.num_attention_heads
NKV = getattr(_tc, "num_key_value_heads", NH)
HD  = getattr(_tc, "head_dim", H // NH)
print(f"MLLM hidden size {H}, layers {L}, attention heads {NH}, kv heads {NKV}, head_dim {HD}"
      f"   (read from {MODEL}/mllm/config.json)")

LORA_R4 = 38 * 16800 * 4   # 38 attention blocks x (to_q,to_k,to_v,to_out.0) = 152 adapted modules; x rank
print(f"LoRA rank 4 trainable parameters: {LORA_R4:,}")

tok = AutoTokenizer.from_pretrained(MODEL, subfolder="processor", trust_remote_code=True)
DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
rows = [_json.loads(l) for l in open(f"{DRIVE}/test.jsonl", encoding="utf-8")]
n = [len(tok(r["instruction"]).input_ids) for r in rows]
INSTR = max(n)
print(f"instruction tokens over {len(n)} test rows: mean {sum(n)/len(n):.0f}, median "
      f"{sorted(n)[len(n)//2]}, max {INSTR}   (measured with the model's own tokenizer)")

print()
for cap, lab in ((888, "training cap, maximum_text_tokens"), (1024, "inference default / thesis operating cap")):
    p = cap - INSTR
    print(f"cap {cap} ({lab}): {p} learnable tokens fit alongside the longest instruction")
    print(f"    shallow variants (i)/(ii) at h={H}: {p*H:,} params = {100*p*H/LORA_R4:.0f}% of LoRA r4")
print(f"\nexact match to LoRA r4 needs {LORA_R4/H:.0f} shallow tokens at h={H} -- above both caps.")
# K and V are num_key_value_heads x head_dim wide under grouped-query attention, not the full
# hidden size. Using H overstates the per-token cost by H / (NKV*HD), which is 8x on this model.
deep = 2 * NKV * HD * L
P_MATCH = round(LORA_R4 / deep)
print(f"deep variant (iii), learnable K,V at every one of {L} layers: "
      f"2 x {NKV} x {HD} x {L} = {deep:,} params/token, {LORA_R4/deep:.1f} tokens to match")
for p in (P_MATCH - 1, P_MATCH, P_MATCH + 1):
    print(f"    p={p:4d}: {p*deep:,} params = {100*p*deep/LORA_R4:.1f}% of LoRA r4")

Mounted at /content/drive


config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

MLLM hidden size 2048, layers 36, attention heads 16, kv heads 2, head_dim 128   (read from OmniGen2/OmniGen2/mllm/config.json)
LoRA rank 4 trainable parameters: 2,553,600


tokenizer_config.json:   0%|          | 0.00/8.01k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

processor/tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

processor/tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

instruction tokens over 488 test rows: mean 149, median 148, max 220   (measured with the model's own tokenizer)

cap 888 (training cap, maximum_text_tokens): 668 learnable tokens fit alongside the longest instruction
    shallow variants (i)/(ii) at h=2048: 1,368,064 params = 54% of LoRA r4
cap 1024 (inference default / thesis operating cap): 804 learnable tokens fit alongside the longest instruction
    shallow variants (i)/(ii) at h=2048: 1,646,592 params = 64% of LoRA r4

exact match to LoRA r4 needs 1247 shallow tokens at h=2048 -- above both caps.
deep variant (iii), learnable K,V at every one of 36 layers: 2 x 2 x 128 x 36 = 18,432 params/token, 138.5 tokens to match
    p= 138: 2,543,616 params = 99.6% of LoRA r4
    p= 139: 2,562,048 params = 100.3% of LoRA r4
    p= 140: 2,580,480 params = 101.1% of LoRA r4


## 9. Summary
- OmniGen2 ships no soft-prompting, prefix-tuning or textual-inversion path; LoRA is the only parameter-efficient recipe provided. The arm is a from-scratch build.
- Variant (i) modifies one call: `self.mllm(text_input_ids, ...)` becomes an `inputs_embeds` call with p learnable rows prepended. Learnable rows traverse the frozen MLLM and can change how the instruction is read.
- Variant (ii) concatenates p learnable rows onto `prompt_embeds` after the MLLM. Rows never traverse the MLLM; they condition the decoder directly. Petrov's result does not cover this path.
- Both are bounded by the same sequence budget if the decoder is to stay inside trained lengths. Exact parameter matching to LoRA rank 4 is not reachable by either at the model's hidden size: it needs P=1,247, past the sequence budget. The constraint is the hidden size, not the injection idea — a third variant added later, prefix K/V at every layer, costs 2 x n_kv_heads x head_dim = 256 per token rather than 2048 and reaches 100.3% of rank 4 at p=139 (soft_prompt_iii_probe).